# 📦 Phase 3 & 4 — Creator Analytics Pipeline

Notebook này thực hiện:
- Phase 3: Xây dựng creator-level dataset (1 creator = 1 dòng)
- Phase 4: Feature selection và chuẩn hoá cho clustering

Output:
- creator_features.csv
- X_profile.npy
- X_profile_performance.npy
- X_full.npy

Mục tiêu cuối:
→ Chuẩn bị dữ liệu cho bài toán **clustering đa chiều (multi-dimensional clustering)** trên TikTok creators

#  🟦 CELL 1 — Setup & Import

In [13]:
!pip install scikit-learn


  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl (8.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached scipy-1.17.1-cp312-cp312-win_amd64.whl (36.5 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ----------------------------- 1/4 [scipy]
   ---------- ------------------

In [14]:
# ==============================
# 1. IMPORT LIBRARIES
# ==============================

## Import thư viện xử lý dữ liệu và scaling
# RobustScaler được dùng vì dữ liệu có outlier mạnh (TikTok views thường lệch phải)

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, RobustScaler

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

✅ Libraries loaded


# 🟦 CELL 2 — Load Data

## 🔹 Step 1 — Load Data & Chuẩn hoá tên cột

Vấn đề:
- Dataset raw sử dụng UPPERCASE (FOLLOWERS, VIEW_COUNT...)
- Pipeline sử dụng lowercase (followers, view...)

Giải pháp:
→ Rename toàn bộ cột về format chuẩn để tránh lỗi KeyError

Ngoài ra:
- Loại bỏ RAW_JSON (không dùng, rất nặng)

In [15]:
# ==============================
# 2. LOAD DATA + RENAME COLUMNS
# ==============================


CREATORS_PATH = "creators.csv" # Đọc dữ liệu từ file CSV
VIDEOS_PATH = "videos.csv"

creators = pd.read_csv(CREATORS_PATH)
videos = pd.read_csv(VIDEOS_PATH)

# ---- DROP RAW JSON (nếu có) ----
# Drop các cột không cần thiết (RAW_JSON thường rất lớn và không dùng)
creators = creators.drop(columns=['RAW_JSON'], errors='ignore')
videos = videos.drop(columns=['RAW_JSON'], errors='ignore')

# ---- RENAME CREATORS ----

# Rename cột để thống nhất naming convention:
# - VIEW_COUNT → view
# - LIKE_COUNT → like
# → giúp pipeline phía sau không bị lỗi


creators = creators.rename(columns={
    'FOLLOWERS': 'followers',
    'FOLLOWING_COUNT': 'following_count',
    'FRIEND_COUNT': 'friend_count',
    'ENGAGEMENT': 'engagement',
    'MEDIAN_VIEWS': 'median_views',
    'VIDEO_COUNT': 'video_count',
    'COLLAB_SCORE': 'collab_score',
    'PRICE': 'price',
    'BROADCAST_SCORE': 'broadcast_score'
})

# ---- RENAME VIDEOS ----
videos = videos.rename(columns={
    'VIEW_COUNT': 'view',
    'LIKE_COUNT': 'like',
    'COMMENT_COUNT': 'comment',
    'SHARE_COUNT': 'share',
    'SAVE_COUNT': 'save',
    'CREATE_TIME': 'create_time'
})
# In ra columns để verify mapping đã đúng
print("Creators columns:", creators.columns.tolist())
print("Videos columns:", videos.columns.tolist())

print("✅ Load + Rename done")

Creators columns: ['CREATOR_ID', 'followers', 'following_count', 'friend_count', 'engagement', 'median_views', 'TOTAL_LIKES', 'DIGG_COUNT', 'video_count', 'collab_score', 'price', 'MISSING_PRICE_FLAG', 'CATEGORY', 'SNAPSHOT_TIME', 'CRAWL_STATUS', 'broadcast_score']
Videos columns: ['VIDEO_ID', 'CREATOR_ID', 'create_time', 'ANCHOR_TYPES', 'view', 'like', 'comment', 'share', 'save', 'VQSCORE', 'BITRATE', 'CATEGORY_TYPE', 'TITLE', 'DESC', 'MUSIC_TITLE', 'MUSIC_AUTHOR', 'SNAPSHOT_TIME', 'MUSIC_PLAY_URL']
✅ Load + Rename done


# 🟦 CELL 3 — Basic Cleaning

## 🔹 Step 2 — Data Cleaning

Mục tiêu:
- Chuẩn hoá dữ liệu trước khi feature engineering

Các bước:
- Convert ID về string (tránh lỗi merge)
- Convert thời gian về datetime
- Loại bỏ video không hợp lệ (view = 0)
- Fill missing values

Lưu ý:
→ Không dùng view = 0 vì sẽ gây lỗi khi tính rate (chia cho 0)

In [16]:
# ==============================
# 3. DATA CLEANING
# ==============================

# Remove duplicate columns
# Loại bỏ duplicate column nếu có (do merge/export lỗi)
videos = videos.loc[:, ~videos.columns.duplicated()]

# Convert ID to string
# Convert ID sang string để đảm bảo join không lỗi
creators['CREATOR_ID'] = creators['CREATOR_ID'].astype(str)
videos['CREATOR_ID'] = videos['CREATOR_ID'].astype(str)

# Convert create_time sang datetime để tính toán thời gian (gap, trend)
# Convert datetime (robust)
if 'create_time' in videos.columns:
    videos['create_time'] = pd.to_datetime(
        videos['create_time'],
        errors='coerce',
        utc=True
    )

# Remove invalid rows
# Loại bỏ video có view = 0 (không hợp lệ cho rate calculation)
videos = videos[videos['view'] > 0]

# Fill NA
# Fill NA để tránh crash trong aggregation
videos = videos.fillna(0)
creators = creators.fillna(0)

print("Videos after cleaning:", videos.shape)
print("Creators after cleaning:", creators.shape)

print("✅ Cleaning done")

Videos after cleaning: (1000, 18)
Creators after cleaning: (1000, 16)
✅ Cleaning done


# PHASE 3 — FEATURE ENGINEERING

# 🟦 CELL 4 — Profile Features

## 🔹 Step 3 — Profile Features (Creator-level)

Mục tiêu:
Mô tả đặc điểm cơ bản của creator:
- Quy mô (followers)
- Hiệu suất tổng thể (median_views)
- Khả năng kiếm tiền (price)

Transform:
- log1p: giảm skew của dữ liệu (rất quan trọng cho clustering)

Feature bổ sung:
- engagement_rate_profile → chất lượng follower
- follower_to_view_ratio → hiệu quả reach

In [17]:
# ==============================
# 4. PROFILE FEATURES
# ==============================

profile = creators.copy()

# Log transform
# Log transform để giảm ảnh hưởng của outlier (followers có thể rất lớn)
profile['log_followers'] = np.log1p(profile['followers'])
profile['log_median_views'] = np.log1p(profile['median_views'])
profile['log_price'] = np.log1p(profile['price'])

# Derived features
# engagement_rate_profile:
# → đo chất lượng follower (engagement trên mỗi follower)
profile['engagement_rate_profile'] = profile['engagement'] / (profile['followers'] + 1)

# price_per_view:
# → đo hiệu quả monetization
profile['price_per_view'] = profile['price'] / (profile['median_views'] + 1)

# follower_to_view_ratio:
# → đo mức độ chuyển đổi follower → view
profile['follower_to_view_ratio'] = profile['median_views'] / (profile['followers'] + 1)

# Select columns
profile_features = profile[[
    'CREATOR_ID',
    'log_followers',
    'log_median_views',
    'log_price',
    'engagement_rate_profile',
    'price_per_view',
    'follower_to_view_ratio',
    'video_count',
    'following_count',
    'friend_count',
    'collab_score',
    'broadcast_score'
]]

print("✅ Profile features done")

✅ Profile features done


# 🟦 CELL 5 — Performance Features

## 🔹 Step 4 — Performance Features (Video → Creator)

Mục tiêu:
Tổng hợp hiệu suất video thành đặc trưng cấp creator

Quan trọng:
- Không dùng raw like/comment → dùng rate
- Ưu tiên median & p90 → robust với outlier

Feature chính:
- median_view → hiệu suất điển hình
- p90_view → hiệu suất cao
- virality_score → khả năng viral
- viral_video_ratio → tần suất viral

In [18]:
# ==============================
# 5. PERFORMANCE FEATURES
# ==============================

df = videos.copy()

# ---- RATE FEATURES ----
# Tính rate để normalize theo view (giảm bias do scale)
df['like_rate'] = df['like'] / df['view']
df['comment_rate'] = df['comment'] / df['view']
df['share_rate'] = df['share'] / df['view']
df['save_rate'] = df['save'] / df['view']

df['engagement_rate'] = (
    df['like'] + df['comment'] + df['share'] + df['save']
) / df['view']

# ---- AGGREGATION ----
# Aggregation theo CREATOR_ID:
# → chuyển từ video-level → creator-level
agg = df.groupby('CREATOR_ID').agg({
    'view': ['median', 'mean', 'max'],
    'like_rate': 'mean',
    'comment_rate': 'mean',
    'share_rate': 'mean',
    'save_rate': 'mean',
    'engagement_rate': 'mean'
})

agg.columns = ['_'.join(col) for col in agg.columns]

# ---- QUANTILES ----
# p90_view:
# → đại diện cho "best performance" nhưng ổn định hơn max
p90 = df.groupby('CREATOR_ID')['view'].quantile(0.9)
p10 = df.groupby('CREATOR_ID')['view'].quantile(0.1)

performance = agg.copy()
performance['p90_view'] = p90
performance['p10_view'] = p10

# ---- VIRALITY ----
# virality_score:
# → đo độ chênh giữa top video và baseline
performance['virality_score'] = performance['p90_view'] / (performance['view_median'] + 1)

# ---- VIRAL VIDEO RATIO ----
# viral_video_ratio:
# → % video vượt ngưỡng viral (2x median)
median_view = df.groupby('CREATOR_ID')['view'].transform('median')
df['is_viral'] = df['view'] > (2 * median_view)

viral_ratio = df.groupby('CREATOR_ID')['is_viral'].mean()
performance['viral_video_ratio'] = viral_ratio

performance = performance.reset_index()

print("Performance shape:", performance.shape)
print("✅ Performance features done")

Performance shape: (449, 13)
✅ Performance features done


# 🟦 CELL 6 — Behavior Features

## 🔹 Step 5 — Behavior Features

Mục tiêu:
Mô tả cách creator hoạt động theo thời gian

Bao gồm:
- Tần suất đăng video
- Độ ổn định performance
- Khoảng cách giữa các lần đăng

Feature quan trọng:
- cv_view → độ biến động
- consistency_score → độ ổn định
- posting_regularity → mức độ đều đặn

In [19]:
# ==============================
# 6. BEHAVIOR FEATURES
# ==============================

df = videos.copy()

# Sort
# Sắp xếp theo thời gian để tính posting pattern
df = df.sort_values(['CREATOR_ID', 'create_time'])

# Posting gap
# gap_days:
# → khoảng cách giữa 2 video liên tiếp
df['prev_time'] = df.groupby('CREATOR_ID')['create_time'].shift(1)
df['gap_days'] = (df['create_time'] - df['prev_time']).dt.days

# ---- AGGREGATION ----


behavior = df.groupby('CREATOR_ID').agg({
    'VIDEO_ID': 'count',
    'view': ['std', 'mean'],
    'gap_days': ['mean', 'std']
})

behavior.columns = ['_'.join(col) for col in behavior.columns]

behavior = behavior.rename(columns={
    'VIDEO_ID_count': 'video_count_crawled',
    'view_std': 'std_view',
    'view_mean': 'avg_view',
    'gap_days_mean': 'gap_mean',
    'gap_days_std': 'gap_std'
})

# ---- DERIVED ----
# cv_view:
# → normalized variance (std / mean)

# consistency_score:
# → càng cao càng ổn định

# posting_regularity:
# → creator đăng đều hay không
behavior['cv_view'] = behavior['std_view'] / (behavior['avg_view'] + 1)
behavior['consistency_score'] = 1 / (1 + behavior['cv_view'])
behavior['posting_regularity'] = 1 / (1 + behavior['gap_std'])

behavior = behavior.reset_index().fillna(0)

print("Behavior shape:", behavior.shape)
print("✅ Behavior features done")

Behavior shape: (449, 9)
✅ Behavior features done


# 🟦 CELL 7 — Merge All

## 🔹 Step 6 — Merge All Features

Mục tiêu:
Kết hợp:
- Profile
- Performance
- Behavior

→ thành 1 bảng duy nhất (1 creator = 1 dòng)

In [20]:
# ==============================
# 7. MERGE FEATURES
# ==============================
# Merge theo CREATOR_ID
creator_features = profile_features.merge(performance, on='CREATOR_ID', how='left')
creator_features = creator_features.merge(behavior, on='CREATOR_ID', how='left')
# Fill NA để tránh lỗi khi đưa vào model/clustering
creator_features = creator_features.fillna(0)
# Kiểm tra shape cuối cùng
print("Final shape:", creator_features.shape)

Final shape: (1000, 32)


In [21]:
print(creator_features.head())
print(creator_features.describe())

     CREATOR_ID  log_followers  log_median_views  log_price  \
0      ._.ehe23      10.454524          8.867991        0.0   
1       .k.c.c5      11.557908          9.249657        0.0   
2  .siudangiu06      11.428467          8.071219        0.0   
3       .thy.em      10.454524          8.556606        0.0   
4      _.doriis      10.477316          9.249657        0.0   

   engagement_rate_profile  price_per_view  follower_to_view_ratio  \
0                 0.000296             0.0                0.204605   
1                 0.000038             0.0                0.099425   
2                 0.000121             0.0                0.034820   
3                 0.000195             0.0                0.149852   
4                 0.000500             0.0                0.292949   

   video_count  following_count  friend_count  ...  virality_score  \
0        396.0             31.0           0.0  ...        0.999910   
1       1484.0             69.0           0.0  ...       13.

🟦 CELL 8 — Save Phase 3 Output

## 🔹 Step 7 — Save Phase 3 Output

Lưu dataset creator-level để:
- tái sử dụng
- debug
- phân tích ngoài notebook

In [22]:
# ==============================
# 8. SAVE CREATOR FEATURES
# ==============================

creator_features.to_csv("creator_features.csv", index=False)

print("✅ Saved creator_features.csv")

✅ Saved creator_features.csv


# PHASE 4 — FEATURE ENGINEERING FOR CLUSTERING

# 🟦 CELL 9 — Feature Selection


Mục tiêu:
Chọn subset feature phù hợp cho clustering

Chia thành 3 nhóm:
- Profile
- Performance
- Behavior

Nguyên tắc:
- Loại bỏ feature trùng nghĩa
- Ưu tiên feature đã normalize (rate, ratio)

In [23]:
# ==============================
# 9. SELECT FEATURES
# ==============================
# Chọn feature theo từng nhóm

# Profile: quy mô + chất lượng
# Performance: hiệu suất video
# Behavior: pattern hoạt động
profile_cols = [
    'log_followers',
    'engagement_rate_profile',
    'follower_to_view_ratio'
]

performance_cols = [
    'view_median',
    'p90_view',
    'virality_score',
    'engagement_rate_mean',
    'viral_video_ratio'
]

behavior_cols = [
    'video_count_crawled',
    'cv_view',
    'posting_regularity',
    'consistency_score'
]

# 🟦 CELL 10 — Create 3 Dataset Versions

## 🔹 Step 8 — Tạo 3 Version Dataset

Mục tiêu:
So sánh hiệu quả clustering theo từng nhóm feature

- V1: chỉ profile
- V2: profile + performance
- V3: full (quan trọng nhất)

In [24]:
# ==============================
# 10. CREATE DATASETS
# ==============================

X_profile = creator_features[profile_cols]
X_profile_perf = creator_features[profile_cols + performance_cols]
X_full = creator_features[profile_cols + performance_cols + behavior_cols]

print("V1:", X_profile.shape)
print("V2:", X_profile_perf.shape)
print("V3:", X_full.shape)

V1: (1000, 3)
V2: (1000, 8)
V3: (1000, 12)


# 🟦 CELL 11 — Scaling

## 🔹 Step 9 — Scaling

Mục tiêu:
Chuẩn hoá dữ liệu trước khi clustering

Sử dụng:
- RobustScaler (phù hợp dữ liệu có outlier)

Lý do:
→ KMeans / clustering rất nhạy với scale

In [25]:
# ==============================
# 11. SCALING
# ==============================
# Scale từng dataset

# Không scale chung để tránh leakage giữa các version
scaler = RobustScaler()

X_profile_scaled = scaler.fit_transform(X_profile)
X_profile_perf_scaled = scaler.fit_transform(X_profile_perf)
X_full_scaled = scaler.fit_transform(X_full)

print("✅ Scaling done")

✅ Scaling done


🟦 CELL 12 — Save Vectors

In [26]:
# ==============================
# 12. SAVE OUTPUT
# ==============================

np.save("X_profile.npy", X_profile_scaled)
np.save("X_profile_performance.npy", X_profile_perf_scaled)
np.save("X_full.npy", X_full_scaled)

print("✅ All vectors saved")

✅ All vectors saved


In [4]:
import numpy as np
data = np.load('X_full.npy', allow_pickle=True)


In [5]:
data.shape

(1000, 12)